[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/sodacore-certified/notebooks/day-07-soda-cloud-contracts.ipynb#scrollTo=a1b2c3d4)

---
# Day 7 · Soda Cloud — Publishing Scan Results, Alerting, and Data Contracts
**certified-journeys / sodacore-certified** · Review · Connecting local scans to Soda Cloud

> **Goal for today:** Configure Soda Cloud integration, understand the data contracts YAML format, and validate a complete data contract programmatically using the local DuckDB connector.


In [ ]:
%pip install -q soda-core-duckdb


## Step 1 · Soda Cloud — What It Adds

Running Soda Core locally gives you pass/fail results in your terminal. **Soda Cloud** layers on top to provide:

| Feature | Local only | With Soda Cloud |
|---|---|---|
| Scan results | Printed to stdout | Stored, queryable, versioned |
| Alerting | None | Email / Slack / PagerDuty |
| Data contracts | Manual review | Enforced and tracked |
| Trend history | None | Time-series charts per check |

To publish results you add a `soda_cloud` block to `configuration.yml`. No code change is needed — the scan engine reads the config and posts results automatically.

```yaml
# configuration.yml — add this block alongside your data_sources section
soda_cloud:
  host: cloud.soda.io
  api_key_id:     YOUR_SODA_API_KEY_ID
  api_key_secret: YOUR_SODA_API_KEY_SECRET
```

> **Note:** In this notebook we use a placeholder API key so every cell runs offline. The pattern is identical for real keys — just swap the placeholders.


In [ ]:
import duckdb, tempfile, pathlib

# ── Create a temp DuckDB database with an orders table ─────────────────────
tmpdir = pathlib.Path(tempfile.mkdtemp())
db_path = str(tmpdir / "ecommerce.duckdb")

conn = duckdb.connect(db_path)
conn.execute("""
    CREATE TABLE orders (
        id          INTEGER PRIMARY KEY,
        customer_id INTEGER NOT NULL,
        amount      DOUBLE  NOT NULL,
        status      VARCHAR NOT NULL,
        created_at  TIMESTAMP NOT NULL
    )
""")
conn.execute("""
    INSERT INTO orders VALUES
        (1, 101, 49.99,  'completed', '2024-01-15 09:00:00'),
        (2, 102, 120.00, 'pending',   '2024-01-16 10:30:00'),
        (3, 103, 75.50,  'completed', '2024-01-16 11:00:00'),
        (4, 101, 30.00,  'cancelled', '2024-01-17 08:45:00'),
        (5, 104, 200.00, 'completed', '2024-01-17 12:00:00')
""")
conn.close()
print(f"Database created at: {db_path}")
print(f"Working directory:   {tmpdir}")


**What just happened?**

- We created a persistent DuckDB file (`ecommerce.duckdb`) in a temp directory.
- **`tmpdir` is used for every file** in this notebook — config, checks, and contract YAMLs all land there so paths are consistent.
- The `orders` table has 5 rows with realistic types: `INTEGER`, `DOUBLE`, `VARCHAR`, `TIMESTAMP`.


## Step 2 · Add `soda_cloud` to `configuration.yml`

The `configuration.yml` file holds **two top-level keys**:

1. `data_sources` — connection details per source name
2. `soda_cloud` — API credentials for publishing results

When both keys are present, every `scan.execute()` call first runs the checks locally, then **pushes** the results payload to Soda Cloud. The scan behaviour is identical — the cloud block is purely additive.

In the cell below we write the config with placeholder credentials. `scan.set_verbose(True)` will show log lines indicating that the cloud push would be attempted — which lets you verify the configuration is wired up without needing real credentials.


In [ ]:
from soda.scan import Scan

# ── Configuration with soda_cloud block (placeholder credentials) ───────────
config_yml = f"""
data_sources:
  ecommerce:
    type: duckdb
    path: "{db_path}"

soda_cloud:
  host: cloud.soda.io
  api_key_id:     YOUR_SODA_API_KEY_ID
  api_key_secret: YOUR_SODA_API_KEY_SECRET
"""

checks_yml = """
checks for orders:
  - row_count > 0
  - missing_count(customer_id) = 0
  - duplicate_count(id) = 0
"""

config_path = tmpdir / "configuration.yml"
checks_path = tmpdir / "checks_basic.yml"
config_path.write_text(config_yml)
checks_path.write_text(checks_yml)

scan = Scan()
scan.set_data_source_name("ecommerce")
scan.add_configuration_yaml_file(str(config_path))
scan.add_sodacl_yaml_file(str(checks_path))
scan.set_verbose(True)   # reveals "Soda Cloud" log lines even with placeholder creds
scan.execute()
print("\n--- SCAN LOGS ---")
print(scan.get_logs_text())


**What just happened?**

- `scan.set_verbose(True)` activates detailed logging; look for lines containing `Soda Cloud` — they confirm the cloud block is being read.
- **With real credentials** those lines would show HTTP responses from `cloud.soda.io`; with placeholders they show auth failure messages.
- The **check results are unaffected** by cloud connectivity — all three checks pass locally regardless.
- `get_logs_text()` is the single string you should write to a log file or CI artifact for audit purposes.


## Step 3 · Data Contracts — Format and Purpose

A **data contract** is a formal YAML document that declares the schema and quality obligations of a dataset. It sits between producer and consumer teams and is enforced by Soda Core on every scan.

The contract YAML has three sections:

| Section | Purpose |
|---|---|
| `dataset` / `datasource` | Identity — which table in which source |
| `columns` | Schema contract — names and data types |
| `checks` | Quality contract — SodaCL check expressions |

```yaml
dataset: orders
datasource: ecommerce
columns:
  - name: id
    data_type: integer
  - name: customer_id
    data_type: integer
  - name: amount
    data_type: double
  - name: status
    data_type: varchar
  - name: created_at
    data_type: timestamp
checks:
  - row_count > 0
  - missing_count(id) = 0
  - freshness(created_at) < 2d
```

> In Soda Cloud, contracts are stored and versioned. A contract violation raises an **incident** and triggers alerting. Locally, we validate them by translating the contract into a `checks for` block and running a scan.


In [ ]:
import json

# ── Full data contract definition ────────────────────────────────────────────
data_contract = {
    "dataset": "orders",
    "datasource": "ecommerce",
    "columns": [
        {"name": "id",          "data_type": "integer",   "nullable": False},
        {"name": "customer_id", "data_type": "integer",   "nullable": False},
        {"name": "amount",      "data_type": "double",    "nullable": False},
        {"name": "status",      "data_type": "varchar",   "nullable": False},
        {"name": "created_at",  "data_type": "timestamp", "nullable": False},
    ],
    "checks": [
        "row_count > 0",
        "missing_count(id) = 0",
        "missing_count(customer_id) = 0",
        "duplicate_count(id) = 0",
        "invalid_count(status) = 0",
        "min(amount) > 0",
    ]
}

# Save contract as JSON for programmatic use
contract_path = tmpdir / "orders_contract.json"
contract_path.write_text(json.dumps(data_contract, indent=2))
print("Data contract written to:", contract_path)
print(json.dumps(data_contract, indent=2))


**What just happened?**

- The contract is stored as a Python dict and serialised to JSON — easy to version-control and diff.
- `nullable: False` expresses a missing-value obligation; we translate that into `missing_count(col) = 0` checks.
- **The `checks` array contains raw SodaCL expressions** — no translation needed when building the `checks for` block.


## Step 4 · Validate the Contract Programmatically

We now write a function `validate_contract(contract, config_path, db_path)` that:

1. Reads the contract dict
2. Builds a SodaCL schema check (`when required column missing`) from the `columns` list
3. Appends all `checks` entries into a single `checks for <dataset>` block
4. Runs the scan and returns a structured result dict

This is the pattern you would use in a CI pipeline or a data orchestrator (Airflow / Prefect) to gate deployments on contract compliance.


In [ ]:
def validate_contract(contract: dict, config_path: pathlib.Path, tmpdir: pathlib.Path) -> dict:
    """Translate a contract dict into SodaCL and run the scan."""
    dataset    = contract["dataset"]
    datasource = contract["datasource"]
    columns    = [c["name"] for c in contract.get("columns", [])]
    col_types  = {c["name"]: c["data_type"] for c in contract.get("columns", [])}
    checks     = contract.get("checks", [])

    # Build required-column list for schema check
    required_cols = "[" + ", ".join(columns) + "]"

    # Build wrong-type sub-block
    type_lines = "\n".join(f"          {col}: {dtype}" for col, dtype in col_types.items())

    # Combine schema check + explicit checks
    check_lines = "\n".join(f"  - {c}" for c in checks)

    contract_checks_yml = f"""
checks for {dataset}:
  - schema:
      fail:
        when required column missing: {required_cols}
      warn:
        when wrong column type:
{type_lines}
{check_lines}
"""

    contract_checks_path = tmpdir / f"{dataset}_contract_checks.yml"
    contract_checks_path.write_text(contract_checks_yml)

    scan = Scan()
    scan.set_data_source_name(datasource)
    scan.add_configuration_yaml_file(str(config_path))
    scan.add_sodacl_yaml_file(str(contract_checks_path))
    scan.execute()

    results = scan.get_scan_results()
    checks_results = results.get("checks", []) if isinstance(results, dict) else []

    passed = sum(1 for c in checks_results if c.get("outcome") == "pass")
    failed = sum(1 for c in checks_results if c.get("outcome") == "fail")
    warned = sum(1 for c in checks_results if c.get("outcome") == "warn")

    return {
        "dataset": dataset,
        "contract_valid": failed == 0,
        "passed": passed,
        "failed": failed,
        "warned": warned,
        "logs": scan.get_logs_text(),
    }


# Use a config WITHOUT soda_cloud (so no auth errors in output)
config_local_yml = f"""
data_sources:
  ecommerce:
    type: duckdb
    path: "{db_path}"
"""
config_local_path = tmpdir / "configuration_local.yml"
config_local_path.write_text(config_local_yml)

result = validate_contract(data_contract, config_local_path, tmpdir)

print("Contract validation result:")
print(f"  Dataset:        {result['dataset']}")
print(f"  Contract valid: {result['contract_valid']}")
print(f"  Passed:  {result['passed']}")
print(f"  Failed:  {result['failed']}")
print(f"  Warned:  {result['warned']}")
print("\n--- Scan logs ---")
print(result["logs"])


**What just happened?**

- `validate_contract` dynamically generates the SodaCL YAML from the contract dict — **the contract is the single source of truth**.
- The schema check block (`fail when required column missing`) catches dropped columns; the `warn when wrong column type` block catches type drift.
- `contract_valid: True` means no FAIL outcomes — warnings are non-blocking by design.
- **Production pattern:** call `validate_contract` in a DAG task before any downstream transformation; raise an exception if `contract_valid` is `False`.


## Step 5 · Writing the Contract as YAML

JSON works for programmatic use, but teams typically version-control contracts as YAML for readability and `git diff` clarity. Here we write the same contract as a canonical YAML file.

```yaml
# orders_contract.yml
dataset: orders
datasource: ecommerce
columns:
  - name: id
    data_type: integer
    nullable: false
  - name: customer_id
    data_type: integer
    nullable: false
  - name: amount
    data_type: double
    nullable: false
  - name: status
    data_type: varchar
    nullable: false
  - name: created_at
    data_type: timestamp
    nullable: false
checks:
  - row_count > 0
  - missing_count(id) = 0
  - missing_count(customer_id) = 0
  - duplicate_count(id) = 0
  - min(amount) > 0
```

Store one contract file per dataset. Link to it from your data catalog and reference it in your pipeline docs.


In [ ]:
contract_yaml_str = """\
dataset: orders
datasource: ecommerce
columns:
  - name: id
    data_type: integer
    nullable: false
  - name: customer_id
    data_type: integer
    nullable: false
  - name: amount
    data_type: double
    nullable: false
  - name: status
    data_type: varchar
    nullable: false
  - name: created_at
    data_type: timestamp
    nullable: false
checks:
  - row_count > 0
  - missing_count(id) = 0
  - missing_count(customer_id) = 0
  - duplicate_count(id) = 0
  - min(amount) > 0
"""

yaml_contract_path = tmpdir / "orders_contract.yml"
yaml_contract_path.write_text(contract_yaml_str)
print("YAML contract written to:", yaml_contract_path)
print()
print(yaml_contract_path.read_text())


**What just happened?**

- The YAML contract is **human-readable** and diff-friendly — ideal for pull request reviews by data consumers.
- Both JSON and YAML versions contain the same information; pick one format and stick to it per team.
- **Next step in a real pipeline:** parse the YAML with `pyyaml`, pass the resulting dict to `validate_contract`, and fail the CI job if `contract_valid` is `False`.


## Step 6 · Soda Cloud Configuration Patterns — Summary

| Scenario | `configuration.yml` change | Behaviour |
|---|---|---|
| Local only (no cloud) | No `soda_cloud` block | Results printed/returned locally |
| Cloud push enabled | Add `soda_cloud` with real credentials | Results pushed to Soda Cloud after scan |
| Cloud push (placeholder creds) | Add `soda_cloud` with `YOUR_*` values | Logs show attempted push; auth error |
| Verbose logging | `scan.set_verbose(True)` | Detailed HTTP + parse logs visible |

**Alerting** is configured in the Soda Cloud UI, not in YAML. Once results are published, you set thresholds on any check and attach notification channels (Slack webhook, PagerDuty key, email).

> **Security note:** Never commit real `api_key_id` / `api_key_secret` values. Use environment variables or a secrets manager and reference them with `${ENV_VAR}` syntax in the YAML.

```yaml
soda_cloud:
  host: cloud.soda.io
  api_key_id:     ${SODA_API_KEY_ID}
  api_key_secret: ${SODA_API_KEY_SECRET}
```


In [ ]:
# Challenge: Write a function that loads an orders_contract.yml file,
# adds a freshness check (freshness(created_at) < 7d) to the checks list,
# and runs validate_contract on the updated contract.
#
# Hint: parse the YAML string manually or use PyYAML.
# Scaffold:

def add_freshness_and_validate(contract_yml_path: pathlib.Path, config_path: pathlib.Path, tmpdir: pathlib.Path) -> dict:
    # 1. Read and parse the contract YAML
    # import yaml; contract = yaml.safe_load(contract_yml_path.read_text())
    
    # 2. Append the freshness check to contract["checks"]
    # contract["checks"].append("freshness(created_at) < 7d")
    
    # 3. Call validate_contract and return the result
    pass


---
## Day 7 key concepts recap
| Concept | What to remember |
|---|---|
| `soda_cloud` block | Add to `configuration.yml` alongside `data_sources`; uses `api_key_id` + `api_key_secret` |
| `set_verbose(True)` | Reveals cloud push log lines for debugging configuration |
| Data contract structure | `dataset`, `datasource`, `columns` (name + data_type), `checks` (SodaCL expressions) |
| Contract enforcement | Translate columns to schema check; translate checks to SodaCL block; run scan |
| Environment variable pattern | Use `${ENV_VAR}` in YAML — never hard-code real API keys |
| Alerting setup | Configured in Soda Cloud UI; triggered by check outcomes from pushed scan results |

> **Tip:** Store one contract YAML per dataset, version it alongside your dbt models or pipeline code, and reference it in CI. A broken contract on a PR should block the merge.

---
## What's next
**Day 8** → Schema Evolution — detecting and handling upstream schema changes with `schema:` checks, column diff extraction, and WARN vs FAIL strategies.

Mark Day 7 complete in your [tracker](../index.html).
